<!-- SPDX-FileCopyrightText: 2026 Mario Gemoll -->
<!-- SPDX-License-Identifier: 0BSD -->

# The shootout

**The analytic expert, ACT, and our flow-matching policy, on the same 100
domain-randomized scenarios.**

Every policy in this project is scored the same way: run it against a *frozen
scenario manifest* and write `run.json` + `episodes.jsonl`. Because the
scenarios are frozen, two runs are **paired** — every arm faced exactly the same
cube poses, targets, lighting draws and physics draws — so the interesting
question is not "which rate is higher" but "which scenarios did they disagree
on, and why".

This notebook does three things:

1. builds the `pap eval-policy-sim` command for each arm, and optionally runs them;
2. reads the resulting runs and reproduces the official comparison
   (`pap compare-policy-evaluations`);
3. goes past what the CLI can print — where in the task each arm dies, how far
   off its placements land, and the video of a scenario the arms disagreed on.

**Nothing here reimplements the evaluation.** Every number comes from the same
package functions the CLI uses; the notebook only arranges and draws them.

## Before you start

- Install and generate the AprilTag textures — see the [README](../README.md).
  Without them no scene compiles.
- `export MUJOCO_GL=egl` (headless) and `export PAP_DATA_ROOT=...`.
- Scoring is about **a minute per scenario per core**. The full suite is 100
  scenarios per arm; the default below is a 6-scenario slice so the notebook
  runs end to end on a laptop. The last section shows how the real thing is
  sharded.
- The expert needs no checkpoint. ACT and the flow policy do — set them in the
  next cell. Arms without a checkpoint are skipped rather than faked.

In [ ]:
from __future__ import annotations

import json
import lzma
import os
import subprocess
import sys
from dataclasses import dataclass, field
from pathlib import Path

REPO = Path.cwd().parent
SUITE = REPO / "config" / "evaluation" / "dr_100_v1.json.xz"
DATA_ROOT = Path(os.environ.get("PAP_DATA_ROOT", Path.home() / "pick-and-place-data"))
RUNS = DATA_ROOT / "evaluations" / "shootout-dr"

# Scenarios per arm. None runs the whole suite -- about 100 minutes an arm on
# one core, which is why the shard recipe at the bottom exists.
SCENARIOS = 6

# Set these to score the learned arms. Both accept a Hugging Face repository id
# or a local path; leave one as None and that arm is skipped.
ACT_CHECKPOINT = None      # e.g. "mariogemoll/pap-act-randomized"
FLOW_REPOSITORY = None     # a repo holding checkpoint-*.pt and an export/ directory

RUN_EVALUATIONS = False    # True actually scores the arms; False reads runs already in RUNS
SAVE_VIDEOS = True         # the policy's own camera frames, two files a scenario

## The suite

`dr_100_v1` is 100 scenarios whose every randomization axis is frozen into the
manifest: the cube pose, the target, the lighting and material draw
(`act_mild_v1`), the camera miscalibration, and the physics draw. Freezing the
*sample* rather than the seed is what makes two arms comparable — they do not
merely face the same distribution, they face the same hundred worlds.

In [ ]:
manifest = json.loads(lzma.open(SUITE).read())
scenarios = manifest["scenarios"]

print(f"{manifest['suite']}: {len(scenarios)} scenarios, schema v{manifest['schema_version']}")
print("randomization preset:", {s["domain_randomization_preset"] for s in scenarios})
print("workspace regions:  ", sorted({s["workspace_region"] for s in scenarios}))

first = scenarios[0]
print(f"\n{first['scenario_id']} draws, for example:")
print("  light intensity ", round(first["domain_randomization_sample"]["light_intensity"], 3))
print("  cube at         ", [round(v, 3) for v in first["source_position_m"]])
print("  target at       ", [round(v, 3) for v in first["target_position_m"]])
print("  physics keys    ", sorted(first["physics_sample"])[:6], "...")

## The three arms

One command per arm, differing only in the controller leaf and its checkpoint.
Everything that defines the experiment — the manifest, the image size, the
appearance — is shared, which is what lets `compare-policy-evaluations` refuse a
comparison between runs that did not face the same task.

In [ ]:
@dataclass
class Arm:
    """One contestant: a controller leaf, its flags, and the colour it is drawn in."""

    leaf: str
    color: str
    flags: list[str] = field(default_factory=list)
    available: bool = True


def flow_checkpoint_flags(repository: str) -> list[str]:
    """Resolve a flow-policy repository to the two paths the leaf wants."""
    from huggingface_hub import snapshot_download

    local = Path(snapshot_download(repo_id=repository))
    checkpoint = sorted(local.glob("checkpoint*.pt"))[-1]
    return ["--checkpoint", str(checkpoint), "--flow-export", str(local / "export")]


# Categorical slots 1-3: identity, fixed per arm, never reassigned by rank.
ARMS = {
    "scripted": Arm(leaf="scripted", color="#2a78d6"),
    "act": Arm(
        leaf="lerobot",
        color="#eb6834",
        flags=["--checkpoint", str(ACT_CHECKPOINT)] if ACT_CHECKPOINT else [],
        available=ACT_CHECKPOINT is not None,
    ),
    "flow": Arm(
        leaf="flow-image",
        color="#1baf7a",
        flags=flow_checkpoint_flags(FLOW_REPOSITORY) if FLOW_REPOSITORY else [],
        available=FLOW_REPOSITORY is not None,
    ),
}


def eval_command(name: str, arm: Arm) -> list[str]:
    command = [
        "pap", "eval-policy-sim", arm.leaf,
        "--manifest", str(SUITE),
        "--output", str(RUNS / name),
    ]
    if SAVE_VIDEOS:
        command.append("--save-videos")
    if SCENARIOS is not None:
        command += ["--limit", str(SCENARIOS)]
    return command + arm.flags


for name, arm in ARMS.items():
    mark = " " if arm.available else "  (skipped: no checkpoint)"
    print(f"{name}:{mark}\n  " + " ".join(eval_command(name, arm)) + "\n")

In [ ]:
if RUN_EVALUATIONS:
    environment = {**os.environ, "MUJOCO_GL": os.environ.get("MUJOCO_GL", "egl")}
    for name, arm in ARMS.items():
        if not arm.available:
            continue
        if (RUNS / name / "run.json").exists():
            print(f"{name}: already scored, leaving it alone")
            continue
        print(f"--- {name} ---")
        subprocess.run(eval_command(name, arm), cwd=REPO, env=environment, check=True)
else:
    print("RUN_EVALUATIONS is False -- reading whatever is already under")
    print(RUNS)

## Reading the runs

`EvaluationRun` is the same loader the comparison command uses. It reassembles
sharded runs, refuses shards that disagree on settings, and refuses two shards
that scored the same scenario twice.

In [ ]:
sys.path.insert(0, str(REPO / "py" / "src"))
from pick_and_place.cli.compare_policy_evaluations import (  # noqa: E402
    METRICS,
    PLACED_TOLERANCE_M,
    EvaluationRun,
    mcnemar,
    wilson_interval,
)

runs = {}
for name in ARMS:
    directory = RUNS / name
    if (directory / "run.json").exists() or list(directory.glob("shard-*/run.json")):
        runs[name] = EvaluationRun(directory)

if not runs:
    raise SystemExit(
        f"No evaluation runs under {RUNS}.\n"
        "Set RUN_EVALUATIONS = True above, or point RUNS at runs you already have."
    )

for name, run in runs.items():
    suite = run.run["scenario_manifest"]
    print(f"{name:<9} {len(run.episodes):>3} scenarios  "
          f"{suite['suite']}  manifest {suite['sha256'][:12]}  "
          f"{run.run['environment']['image_width']}x{run.run['environment']['image_height']}")

shared = sorted(set.intersection(*(set(run.scenario_ids) for run in runs.values())))
print(f"\n{len(shared)} scenarios common to every arm -- the paired sample.")

## The official answer first

Before drawing anything, the number the project actually quotes. `success` is
the promotion metric: placed within 4 cm, released, *and* settled.
`placed_6cm` and `cube_lifted` are the coarser instruments used to rank
checkpoints that all score near zero on the headline.

In [ ]:
present = [str(RUNS / name) for name in runs]
print(subprocess.run(
    ["pap", "compare-policy-evaluations", *present],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout)

### The same rates, with intervals

A Wilson interval, not a normal approximation: at six or a hundred scenarios the
rates sit near 0 and 1, where the normal approximation runs off the end of the
scale. Read a 6-scenario slice as a smoke test — the intervals will be almost
the full width of the axis, and they are telling the truth about how little six
scenarios say.

In [ ]:
import pandas as pd  # noqa: E402

rows = []
for name, run in runs.items():
    row = {"arm": name, "n": len(run.episodes)}
    for metric in METRICS:
        outcomes = run.outcomes(metric)
        count = sum(outcomes.values())
        low, high = wilson_interval(count, len(outcomes))
        row[metric] = f"{count}/{len(outcomes)} = {count / len(outcomes):.0%}"
        row[f"{metric} 95% CI"] = f"[{low:.0%}, {high:.0%}]"
    rows.append(row)

headline = pd.DataFrame(rows).set_index("arm")
headline

## Who beat whom, on the same scenarios

An unpaired difference of two rates throws away the pairing the frozen manifest
bought. McNemar's test looks only at the scenarios where the two arms
*disagreed* — the ones that carry information about which is better — and gives
an exact two-sided p at p = 0.5.

With a handful of scenarios no p-value here will be small. That is the correct
result, not a broken cell.

In [ ]:
baseline_name = "scripted" if "scripted" in runs else next(iter(runs))
baseline = runs[baseline_name]

comparisons = []
for name, run in runs.items():
    if name == baseline_name:
        continue
    test = mcnemar(run.outcomes("success"), baseline.outcomes("success"), shared)
    comparisons.append({
        "arm": name,
        f"beat {baseline_name} on": test["only_first"],
        f"lost to {baseline_name} on": test["only_second"],
        "both": test["both"],
        "neither": test["neither"],
        "p (exact)": round(test["p_value_exact"], 4),
    })

if not comparisons:
    print(f"Only {baseline_name} has a run; nothing to pair it against.")

pd.DataFrame(comparisons).set_index("arm") if comparisons else None

## Where each arm dies

Success is one bit at the end of a long task. The milestones say *how far* an
arm got, which is the difference between "cannot see the cube" and "can pick it
up but drops it on the way".

**These latch independently; they do not nest,** so the bars need not decrease
down the list. `cube_released` latches at the moment a grasp *ends*, while
`cube_settled` latches whenever a lifted cube comes to rest — so a run can settle
more cubes than it is recorded as releasing.

In [ ]:
import matplotlib as mpl  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402

mpl.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c9c8c3",
    "axes.labelcolor": "#3d3d38",
    "axes.axisbelow": True,
    "text.color": "#1a1a19",
    "xtick.color": "#6b6a63",
    "ytick.color": "#6b6a63",
    "grid.color": "#e8e7e2",
    "grid.linewidth": 0.8,
    "legend.frameon": False,
})

MILESTONES = [
    "pickup_contact_attempted",
    "cube_lifted",
    "stable_carry",
    "target_reached_while_holding",
    "cube_released",
    "cube_settled",
    "successful_placement",
]


def milestone_rate(run, milestone):
    return sum(e["milestones"][milestone] for e in run.episodes) / len(run.episodes)


milestones = pd.DataFrame(
    {name: [milestone_rate(run, m) for m in MILESTONES] for name, run in runs.items()},
    index=[m.replace("_", " ") for m in MILESTONES],
)

positions = range(len(MILESTONES))
height = 0.8 / len(runs)
figure, axes = plt.subplots(figsize=(7.5, 4.2))
for index, (name, run) in enumerate(runs.items()):
    offset = (index - (len(runs) - 1) / 2) * height
    bars = axes.barh(
        [p + offset for p in positions], milestones[name], height=height * 0.9,
        color=ARMS[name].color, label=name,
    )
    axes.bar_label(bars, labels=[f"{v:.0%}" for v in milestones[name]],
                   padding=3, fontsize=8, color="#6b6a63")

axes.set_yticks(list(positions), milestones.index)
axes.invert_yaxis()
axes.set_xlim(0, 1.12)
axes.set_xlabel("share of scenarios reaching this milestone")
axes.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes.grid(axis="x")
axes.grid(axis="y", visible=False)
if len(runs) > 1:
    axes.legend(loc="lower right")
axes.set_title("Which milestones each arm reached", loc="left", pad=12)
plt.tight_layout()
plt.show()

milestones.style.format("{:.0%}")

## Why the failures happened

The oracle's failure flags are not exclusive — an episode can miss the pickup
*and* time out — so these columns do not sum to the scenario count.

In [ ]:
FAILURES = [
    "missed_pickup",
    "unstable_or_lost_grasp",
    "early_release",
    "off_target_placement",
    "unexpected_collision",
    "cube_out_of_bounds",
    "timeout",
]

taxonomy = pd.DataFrame(
    {
        name: [sum(e["failures"][f] for e in run.episodes) for f in FAILURES]
        for name, run in runs.items()
    },
    index=[f.replace("_", " ") for f in FAILURES],
)
taxonomy.loc["episodes"] = [len(run.episodes) for run in runs.values()]
taxonomy

## How close the misses came

Success needs 4 cm. This is the whole distribution of final cube-to-target
distance, so a policy that lands consistently at 5 cm — scoring zero — is
visibly different from one that never gets the cube near the plate.

In [ ]:
figure, axes = plt.subplots(figsize=(7.5, 3.8))
for name, run in runs.items():
    errors = sorted(e["final_xy_error_m"] * 100 for e in run.episodes)
    share = [(index + 1) / len(errors) for index in range(len(errors))]
    axes.step(
        [0, *errors], [0, *share], where="post",
        color=ARMS[name].color, linewidth=2, label=name,
    )

axes.axvline(4.0, color="#6b6a63", linewidth=1, linestyle=":")
axes.annotate("4 cm: success", (4.0, 0.04), xytext=(-6, 0), textcoords="offset points",
              ha="right", fontsize=8, color="#6b6a63")
axes.axvline(PLACED_TOLERANCE_M * 100, color="#6b6a63", linewidth=1, linestyle=":")
axes.annotate("6 cm: placed", (PLACED_TOLERANCE_M * 100, 0.14), xytext=(6, 0),
              textcoords="offset points", fontsize=8, color="#6b6a63")

axes.set_xlabel("final cube-to-target distance (cm)")
axes.set_ylabel("share of scenarios")
axes.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes.set_ylim(0, 1.02)
axes.grid(axis="y")
if len(runs) > 1:
    axes.legend(loc="lower right")
axes.set_title("Where the cube ended up", loc="left", pad=12)
plt.tight_layout()
plt.show()

## Watch a disagreement

The most informative episode is one the arms did not agree on. Below: the
scenarios the baseline solved and a challenger did not, with the policy's own
camera feeds for one of them — `--save-videos` writes the exact frames the
controller was given, not a prettier render.

In [ ]:
from IPython.display import Video, display  # noqa: E402

interesting = None
if len(runs) == 1:
    print(f"Only {baseline_name} ran, so there is nothing to disagree with.")

for name, run in runs.items():
    if name == baseline_name:
        continue
    test = mcnemar(run.outcomes("success"), baseline.outcomes("success"), shared)
    print(f"{name} failed where {baseline_name} succeeded: {test['only_second_scenarios']}")
    print(f"{name} succeeded where {baseline_name} failed: {test['only_first_scenarios']}\n")
    interesting = interesting or next(iter(test["only_second_scenarios"]), None)

# Fall back to the baseline's own worst placement when every arm agreed.
if interesting is None:
    interesting = max(baseline.episodes, key=lambda e: e["final_xy_error_m"])["scenario_id"]
    print(f"Showing {baseline_name}'s worst placement instead.")

print(f"scenario: {interesting}")
for name, run in runs.items():
    video = RUNS / name / "videos" / f"{interesting}-overhead.mp4"
    if video.exists():
        print(f"\n{name} -- overhead")
        display(Video(str(video), embed=True, width=420))

## Running it at full size

Six scenarios is a smoke test. The real number is the whole suite, and a hundred
scenarios an arm is worth sharding across workers or GPUs:

```sh
# one shard per worker, disjoint slices of the same suite
for offset in 0 25 50 75; do
  pap eval-policy-sim flow-image \
    --manifest config/evaluation/dr_100_v1.json.xz \
    --checkpoint "$CHECKPOINT" --flow-export "$EXPORT" \
    --offset "$offset" --limit 25 \
    --output "$PAP_DATA_ROOT/evaluations/shootout-dr/flow/shard-$offset" &
done
wait

pap compare-policy-evaluations \
  "$PAP_DATA_ROOT"/evaluations/shootout-dr/{scripted,act,flow} \
  --baseline "$PAP_DATA_ROOT/evaluations/shootout-dr/scripted"
```

`EvaluationRun` picks up `shard-*/run.json` on its own, and refuses shards that
disagree on any setting that defines the experiment — so a shard launched with
the wrong flag fails the comparison instead of being quietly averaged in.

Two things worth doing before quoting whatever comes out:

- **Score the canonical suite too.** `canonical_100_v1` is the same size with
  randomization off. The gap between the two is what domain randomization cost —
  or bought — for each arm.
- **Do not select a checkpoint on the suite you then quote.** The project keeps
  `heldout_256_v1` untouched for exactly that reason, and has been burned by
  lucky small draws before: several long-quoted numbers turned out to be
  256-scenario flukes worth about seven points.